## FX Rate Analysis and Forecasting with BISTRO

This notebook adapts the **BISTRO** (BIS Time-series Regression Oracle) framework for
**foreign exchange (FX) rate analysis and forecasting**.

**Key differences from the macro (CPI) notebooks:**
- No year-over-year transformation — FX rates are used directly as price levels
- `fx_preprocessing_util.py` provides FX-specific helpers (log returns, rate differential)
- Interest rate differential (US − Euro) is used as a covariate (uncovered interest parity)

**Workflow:**
1. Load monthly EUR/USD exchange rate data
2. Explore historical data and the US–Euro interest rate differential
3. Run a **univariate** BISTRO forecast (FX rate only)
4. Run a **multivariate** BISTRO forecast (FX rate + rate differential)
5. Compare forecast accuracy (RMSE) against an AR(1) baseline

### Data
- Target: `data/sample_eurusd_m.csv` — synthetic EUR/USD monthly rate (2010–2024)
- Covariate: BIS central bank policy rates (US and Euro area) from `data/`

> **Use your own data**: replace `sample_eurusd_m.csv` with any two-column CSV
> (Date, Rate) to forecast a different currency pair.

### Step 1 – Setup
- Make the project code in `src/` available to the notebook.
- Import libraries and helper functions.

### Google Colab users
Colab may preinstall **NumPy 2.x**. The cell below downgrades it automatically.

In [ ]:
import os
import subprocess

try:
    import numpy as np
    if np.__version__.startswith('2.'):
        print(f'Current NumPy is {np.__version__}. Downgrading to 1.26.4...')
        subprocess.run([
            'pip', 'install', '-q', '--force-reinstall',
            'numpy==1.26.4', 'pandas==2.1.4', 'scipy==1.11.4',
        ])
        print('Install complete. Restarting runtime...')
        os.kill(os.getpid(), 9)
    else:
        print(f'Using NumPy {np.__version__}')
except ImportWarning:
    pass

In [ ]:
import os
import sys

if 'google.colab' in sys.modules:
    print('Running in Google Colab. Setting up repository...')
    !git clone -q https://github.com/bis-med-it/bistro.git
    if 'uni2ts' not in sys.modules:
        print('Installing dependencies (this may take a moment)...')
        !pip install -q -r /content/bistro/requirements.txt > /dev/null 2>&1
        print('Installation complete!')

### Restart Colab to finish setup
**Runtime → Restart session**, then continue from the next cell.

In [ ]:
import os
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    os.chdir('/content/bistro/script')

repo_root = Path('..').resolve()
src_root  = Path('../src').resolve()
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule

from inference_util import plot_publication_forecast_comparison, ar1_forecast
from preprocessing_util import (
    aggregate_daily_forecast_to_monthly,
    prepare_long_df_monthly_for_daily_inference,
)
from fx_preprocessing_util import (
    compute_log_returns,
    compute_rate_differential,
    prepare_fx_monthly_for_inference,
)

### Step 2 – Configuration

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `PDT` | 12 | Forecast horizon (months) |
| `CTX` | 120 | Context window (10 years of history) |
| `PSZ` | 32 | Patch size in days (one month ≈ 32 days) |
| `ROLLING_WINDOWS` | 4 | Number of backtesting windows |
| `WINDOW_DISTANCE` | 2 | Gap between consecutive windows (months) |

In [ ]:
MODEL_REPO = repo_root / 'bistro-finetuned'

FREQ             = 'M'            # monthly data
PDT              = 12             # forecast 12 months ahead
CTX              = 120            # 10 years of context
PSZ              = 32             # patch size (days per period)
BSZ              = 32             # batch size
ROLLING_WINDOWS  = 4              # backtesting windows
WINDOW_DISTANCE  = 2              # months between windows

FORECAST_START_DATE = '2023-01-01'
PAIR = 'EURUSD'                   # label for plots

### Step 3 – Data Loading

- **Target**: EUR/USD monthly exchange rate
- **Covariates**: US and Euro area central bank policy rates → interest rate differential

In [ ]:
# --- EUR/USD exchange rate ---
fx_file   = repo_root / 'data' / 'sample_eurusd_m.csv'
df_fx     = pd.read_csv(fx_file, index_col=0)
df_fx.index = pd.to_datetime(df_fx.index).to_period(freq=FREQ)
target_col = df_fx.columns[0]   # 'EURUSD'

# --- Central bank policy rates ---
df_us_rate = pd.read_csv(repo_root / 'data' / 'bis_cbpol_us_m.csv', index_col=0)
df_us_rate.index = pd.to_datetime(df_us_rate.index).to_period(freq=FREQ)

df_eu_rate = pd.read_csv(repo_root / 'data' / 'bis_cbpol_xm_m.csv', index_col=0)
df_eu_rate.index = pd.to_datetime(df_eu_rate.index).to_period(freq=FREQ)

# --- US minus Euro interest rate differential ---
rate_diff  = compute_rate_differential(
    df_us_rate.iloc[:, 0],
    df_eu_rate.iloc[:, 0],
    name='rate_diff',
)
df_rate_diff = rate_diff.to_frame()

print(f'EUR/USD:         {df_fx.index[0]} → {df_fx.index[-1]} ({len(df_fx)} months)')
print(f'Rate diff (US−EU): {df_rate_diff.index[0]} → {df_rate_diff.index[-1]} ({len(df_rate_diff)} months)')

df_fx.tail(3)

In [ ]:
# Historical EUR/USD and interest rate differential
common_idx    = df_rate_diff.index.intersection(df_fx.index)
df_rd_common  = df_rate_diff.loc[common_idx]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax1.plot(df_fx.index.to_timestamp(), df_fx[target_col], color='black', lw=1.4)
ax1.set_ylabel('EUR/USD')
ax1.set_title('EUR/USD Exchange Rate and US − Euro Interest Rate Differential')
ax1.grid(True, alpha=0.3)
ax1.spines[['top', 'right']].set_visible(False)

ax2.plot(df_rd_common.index.to_timestamp(), df_rd_common['rate_diff'], color='C1', lw=1.4)
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_ylabel('US − Euro Rate (%)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Monthly log-return statistics
log_ret = compute_log_returns(df_fx[target_col])

stats = pd.DataFrame({
    'mean (%)':   [round(float(log_ret.mean()), 4)],
    'std (%)':    [round(float(log_ret.std()),  4)],
    'min (%)':    [round(float(log_ret.min()),  4)],
    'max (%)':    [round(float(log_ret.max()),  4)],
    'ann. vol (%)': [round(float(log_ret.std() * (12 ** 0.5)), 4)],
})
print(f'Monthly log-return statistics for {target_col}')
stats

### Step 4 – Univariate Forecast (EUR/USD only)

BISTRO forecasts the EUR/USD level using only historical exchange rate data.
We run a rolling-origin backtest with `ROLLING_WINDOWS` windows starting from
`FORECAST_START_DATE`, then compare against an AR(1) baseline.

In [ ]:
# Prepare univariate data for daily Moirai inference
prep = prepare_fx_monthly_for_inference(
    df_fx,
    target_col=target_col,
    freq=FREQ,
    use_log_returns=False,       # forecast raw EUR/USD levels
    forecast_start_date=FORECAST_START_DATE,
    pdt_patches=PDT,
    ctx_patches=CTX,
    steps_per_period=PSZ,
    rolling_windows=ROLLING_WINDOWS,
    window_distance_patches=WINDOW_DISTANCE,
)

if prep.windows < 1:
    raise ValueError(
        f'Not enough data after {prep.train_end} to create a window '
        f'(test_len={(prep.df_yoy_dt.index > prep.cutoff_date_dt).sum()}, PDT={PDT}).'
    )

print(f'Windows: {prep.windows}  |  PDT steps: {prep.pdt_steps}  |  CTX steps: {prep.ctx_steps}')

ds = PandasDataset(prep.daily_df, target=target_col)
train, test_template = split(ds, date=prep.cutoff_period_daily)

test_data = test_template.generate_instances(
    prediction_length=prep.pdt_steps,
    windows=prep.windows,
    distance=prep.dist_steps,
    max_history=prep.ctx_steps,
)

In [ ]:
# Load pretrained BISTRO and generate probabilistic forecasts
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(str(MODEL_REPO)),
    prediction_length=prep.pdt_steps,
    context_length=prep.ctx_steps,
    patch_size=PSZ,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=0,
    past_feat_dynamic_real_dim=0,
)

predictor  = model.create_predictor(batch_size=BSZ)
inputs     = list(test_data.input)
labels     = list(test_data.label)
forecasts  = list(predictor.predict(test_data.input))

print(f'Generated {len(forecasts)} forecast windows.')

In [ ]:
# Aggregate daily forecast samples → monthly, compute RMSE vs AR(1)
bistro_univ_by_window = {}
rmse_univ_rows        = []

for w in range(prep.windows):
    samples      = np.asarray(forecasts[w].samples, dtype=float)
    label_target = np.asarray(labels[w]['target'],  dtype=float)
    inp_target   = np.asarray(inputs[w]['target'],  dtype=float)
    last_input   = float(inp_target[-1]) if inp_target.size > 0 else None

    preds, _, ci = aggregate_daily_forecast_to_monthly(
        samples, label_target, last_input,
        steps_per_period=PSZ, expected_periods=PDT,
    )

    pred_index = pd.period_range(
        start=prep.forecast_start + w * WINDOW_DISTANCE,
        periods=PDT, freq=FREQ,
    )

    dfw = pd.DataFrame(
        {'bistro_pred': preds, 'bistro_lo': ci[:, 0], 'bistro_hi': ci[:, 1]},
        index=pred_index,
    )

    # AR(1) baseline fitted on CTX months before each window
    train_end_w = pred_index[0] - 1
    train_y     = prep.df_monthly[target_col].loc[:train_end_w].tail(CTX).astype(float)
    try:
        ar1_pred = ar1_forecast(
            train_y, pred_index, method='statsmodels', trend='c', validate_index=True,
        )
    except Exception:
        ar1_pred = pd.Series(np.nan, index=pred_index)
    dfw['ar1_pred'] = ar1_pred

    bistro_univ_by_window[w] = dfw

    actual   = prep.df_monthly[target_col].reindex(pred_index).astype(float)
    valid_b  = actual.notna() & dfw['bistro_pred'].notna()
    valid_a  = actual.notna() & ar1_pred.notna()

    rmse_b = float(np.sqrt(np.mean((dfw['bistro_pred'][valid_b] - actual[valid_b]) ** 2))) if valid_b.any() else np.nan
    rmse_a = float(np.sqrt(np.mean((ar1_pred[valid_a]          - actual[valid_a]) ** 2))) if valid_a.any() else np.nan
    r_rmse = round(rmse_b / rmse_a, 4) if (np.isfinite(rmse_b) and np.isfinite(rmse_a) and rmse_a != 0) else np.nan

    rmse_univ_rows.append({
        'window':      w,
        'test_start':  pred_index[0],
        'test_end':    pred_index[-1],
        'rmse_bistro': round(rmse_b, 4),
        'rmse_ar1':    round(rmse_a, 4),
        'r_rmse':      r_rmse,
        'n_valid':     int(valid_b.sum()),
    })

rmse_univ_table = pd.DataFrame(rmse_univ_rows)
print('=== Univariate BISTRO — RMSE per window ===')
rmse_univ_table

In [ ]:
# Plot: actual EUR/USD vs BISTRO forecast vs AR(1) — first window
w = 0
forecast_start_w = prep.forecast_start + w * WINDOW_DISTANCE

df_actual = prep.df_monthly[[target_col]].rename(columns={target_col: 'actual'})
df_pred   = bistro_univ_by_window[w]

plot_from = prep.forecast_start - min(CTX, 60)
plot_to   = df_pred.index.max()

df_plot = df_actual.join(df_pred[['bistro_pred', 'ar1_pred']], how='outer').sort_index()
df_plot = df_plot.loc[plot_from:plot_to]

out_dir = repo_root / 'script' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plot_publication_forecast_comparison(
    df_plot,
    actual_col='actual',
    forecast_cols={
        'bistro_pred': 'BISTRO (median)',
        'ar1_pred':    'AR(1)',
    },
    forecast_start=forecast_start_w,
    title=f'{PAIR} — Univariate Forecast (window {w})',
    ylabel='EUR/USD',
    savepaths=[out_dir / 'forecast_fx_univariate.png'],
)
fig

### Step 5 – Multivariate Forecast with Interest Rate Differential

We add the **US − Euro interest rate differential** as a past covariate.
Economically, when US rates exceed Euro rates the dollar tends to strengthen
(EUR/USD falls), and vice versa — capturing this via Moirai's any-variate
attention may improve forecasts.

The rate differential is `past_feat_dynamic_real`: observed up to the
training cutoff but not available in the forecast horizon.

In [ ]:
# Merge EUR/USD + rate differential into long-format dataframe
df_mv = df_fx[[target_col]].rename(columns={target_col: 'target'}).copy()
df_mv['item_id'] = PAIR
df_mv = df_mv.join(df_rate_diff, how='inner')   # inner join on common dates

print(f'Merged dataframe: {df_mv.index[0]} → {df_mv.index[-1]} ({len(df_mv)} months)')
print(f'Columns: {df_mv.columns.tolist()}')

prep_mv = prepare_long_df_monthly_for_daily_inference(
    df_mv,
    item_id_col='item_id',
    target_col='target',
    past_dynamic_real_cols=['rate_diff'],
    freq=FREQ,
    forecast_start_date=FORECAST_START_DATE,
    pdt_patches=PDT,
    ctx_patches=CTX,
    steps_per_period=PSZ,
    rolling_windows=ROLLING_WINDOWS,
    window_distance_patches=WINDOW_DISTANCE,
)

if prep_mv.windows < 1:
    raise ValueError('Not enough multivariate data for a backtesting window.')

print(f'\nMultivariate windows: {prep_mv.windows}')
prep_mv.daily_long_df.head(3)

In [ ]:
# Build GluonTS dataset with rate_diff as a past covariate
ds_mv = PandasDataset.from_long_dataframe(
    prep_mv.daily_long_df,
    item_id='item_id',
    past_feat_dynamic_real=['rate_diff'],
    feat_dynamic_real=[],
)

train_mv, test_template_mv = split(ds_mv, date=prep_mv.cutoff_period_daily)

test_data_mv = test_template_mv.generate_instances(
    prediction_length=prep_mv.pdt_steps,
    windows=prep_mv.windows,
    distance=prep_mv.dist_steps,
    max_history=prep_mv.ctx_steps,
)

print(f'past_feat_dynamic_real dim: {ds_mv.num_past_feat_dynamic_real}')

In [ ]:
# Load BISTRO and set past_feat_dynamic_real_dim=1 for the rate differential
model_mv = MoiraiForecast(
    module=MoiraiModule.from_pretrained(str(MODEL_REPO)),
    prediction_length=prep_mv.pdt_steps,
    context_length=prep_mv.ctx_steps,
    patch_size=PSZ,
    num_samples=100,
    target_dim=1,
    feat_dynamic_real_dim=ds_mv.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds_mv.num_past_feat_dynamic_real,
)

predictor_mv  = model_mv.create_predictor(batch_size=BSZ)
inputs_mv     = list(test_data_mv.input)
labels_mv     = list(test_data_mv.label)
forecasts_mv  = list(predictor_mv.predict(test_data_mv.input))

print(f'Generated {len(forecasts_mv)} multivariate forecast windows.')

In [ ]:
# Aggregate multivariate forecasts + compute RMSE vs AR(1)
bistro_mv_by_window = {}
rmse_mv_rows        = []

for w in range(prep_mv.windows):
    samples      = np.asarray(forecasts_mv[w].samples, dtype=float)
    label_target = np.asarray(labels_mv[w]['target'],  dtype=float)
    inp_target   = np.asarray(inputs_mv[w]['target'],  dtype=float)
    last_input   = float(inp_target[-1]) if inp_target.size > 0 else None

    preds, _, ci = aggregate_daily_forecast_to_monthly(
        samples, label_target, last_input,
        steps_per_period=PSZ, expected_periods=PDT,
    )

    pred_index = pd.period_range(
        start=prep_mv.forecast_start + w * WINDOW_DISTANCE,
        periods=PDT, freq=FREQ,
    )

    dfw = pd.DataFrame(
        {'bistro_mv_pred': preds, 'bistro_mv_lo': ci[:, 0], 'bistro_mv_hi': ci[:, 1]},
        index=pred_index,
    )

    train_end_w = pred_index[0] - 1
    train_y     = prep_mv.df_monthly_target['target'].loc[:train_end_w].tail(CTX).astype(float)
    try:
        ar1_pred = ar1_forecast(
            train_y, pred_index, method='statsmodels', trend='c', validate_index=True,
        )
    except Exception:
        ar1_pred = pd.Series(np.nan, index=pred_index)
    dfw['ar1_pred'] = ar1_pred

    bistro_mv_by_window[w] = dfw

    actual   = prep_mv.df_monthly_target['target'].reindex(pred_index).astype(float)
    valid_b  = actual.notna() & dfw['bistro_mv_pred'].notna()
    valid_a  = actual.notna() & ar1_pred.notna()

    rmse_b = float(np.sqrt(np.mean((dfw['bistro_mv_pred'][valid_b] - actual[valid_b]) ** 2))) if valid_b.any() else np.nan
    rmse_a = float(np.sqrt(np.mean((ar1_pred[valid_a]             - actual[valid_a]) ** 2))) if valid_a.any() else np.nan
    r_rmse = round(rmse_b / rmse_a, 4) if (np.isfinite(rmse_b) and np.isfinite(rmse_a) and rmse_a != 0) else np.nan

    rmse_mv_rows.append({
        'window':         w,
        'test_start':     pred_index[0],
        'test_end':       pred_index[-1],
        'rmse_bistro_mv': round(rmse_b, 4),
        'rmse_ar1':       round(rmse_a, 4),
        'r_rmse':         r_rmse,
        'n_valid':        int(valid_b.sum()),
    })

rmse_mv_table = pd.DataFrame(rmse_mv_rows)
print('=== Multivariate BISTRO — RMSE per window ===')
rmse_mv_table

In [ ]:
# Plot: actual EUR/USD vs multivariate BISTRO vs AR(1)
# Rate differential shown on a secondary axis (training period only)
w = 0
forecast_start_w = prep_mv.forecast_start + w * WINDOW_DISTANCE
train_end_w      = forecast_start_w - 1

df_actual_mv = prep_mv.df_monthly_target[['target']].rename(columns={'target': 'actual'})
df_pred_mv   = bistro_mv_by_window[w]

plot_from = prep_mv.forecast_start - min(CTX, 60)
plot_to   = df_pred_mv.index.max()

df_plot_mv = df_actual_mv.join(
    df_pred_mv[['bistro_mv_pred', 'ar1_pred']], how='outer'
).sort_index()
df_plot_mv = df_plot_mv.loc[plot_from:plot_to]

fig, ax = plot_publication_forecast_comparison(
    df_plot_mv,
    actual_col='actual',
    forecast_cols={
        'bistro_mv_pred': f'BISTRO + Rate Diff (median)',
        'ar1_pred':       'AR(1)',
    },
    forecast_start=forecast_start_w,
    title=f'{PAIR} — Multivariate Forecast with Interest Rate Differential (window {w})',
    ylabel='EUR/USD',
    savepaths=[repo_root / 'script' / 'figures' / 'forecast_fx_multivariate.png'],
)

# Overlay rate differential on a secondary y-axis (up to training cutoff)
df_rd_train = df_rate_diff.loc[
    df_rate_diff.index.intersection(
        pd.period_range(plot_from, train_end_w, freq=FREQ)
    )
]
ax2 = ax.twinx()
ax2.plot(
    df_rd_train.index.to_timestamp(),
    df_rd_train['rate_diff'].to_numpy(dtype=float),
    color='C3', lw=1.2, alpha=0.65, label='US−EU Rate Diff',
)
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.set_ylabel('Interest Rate Diff (%)', color='C3')
ax2.tick_params(axis='y', colors='C3')
ax2.spines['right'].set_color('C3')
ax2.yaxis.label.set_color('C3')

h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc='upper left', frameon=False, ncol=2)
fig.tight_layout()
fig

### Step 6 – Forecast Accuracy Summary

Compare **univariate BISTRO** vs **multivariate BISTRO** (+ rate differential)
vs the **AR(1)** baseline.

- **RMSE**: root mean squared error in EUR/USD units
- **R-RMSE**: relative RMSE vs AR(1) — values below 1.0 beat the baseline

In [ ]:
n_windows = min(len(rmse_univ_rows), len(rmse_mv_rows))

df_compare = pd.DataFrame({
    'window':           [r['window']      for r in rmse_univ_rows[:n_windows]],
    'test_start':       [r['test_start']  for r in rmse_univ_rows[:n_windows]],
    'test_end':         [r['test_end']    for r in rmse_univ_rows[:n_windows]],
    'BISTRO_Univ_RMSE': [r['rmse_bistro'] for r in rmse_univ_rows[:n_windows]],
    'BISTRO_MV_RMSE':   [r['rmse_bistro_mv'] for r in rmse_mv_rows[:n_windows]],
    'AR1_RMSE':         [r['rmse_ar1']    for r in rmse_univ_rows[:n_windows]],
    'R-RMSE_Univ':      [r['r_rmse']      for r in rmse_univ_rows[:n_windows]],
    'R-RMSE_MV':        [r['r_rmse']      for r in rmse_mv_rows[:n_windows]],
})

print('=== Forecast Accuracy Comparison (lower RMSE = better) ===')
df_compare

In [ ]:
# Bar chart: RMSE by window and model
x    = df_compare['window'].tolist()
w_b  = 0.25
xpos = range(len(x))

fig, ax = plt.subplots(figsize=(9, 4))

ax.bar([p - w_b   for p in xpos], df_compare['BISTRO_Univ_RMSE'], width=w_b, label='BISTRO Univariate', color='C0')
ax.bar([p          for p in xpos], df_compare['BISTRO_MV_RMSE'],   width=w_b, label='BISTRO + Rate Diff', color='C2')
ax.bar([p + w_b   for p in xpos], df_compare['AR1_RMSE'],          width=w_b, label='AR(1) baseline',     color='C3', alpha=0.7)

ax.set_xticks(list(xpos))
ax.set_xticklabels([f"W{r}\n{df_compare['test_start'][r]}" for r in xpos], fontsize=9)
ax.set_ylabel('RMSE (EUR/USD)')
ax.set_title(f'{PAIR} Forecast Accuracy — BISTRO vs AR(1) by Window')
ax.legend(frameon=False)
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'forecast_fx_rmse_comparison.png', dpi=150, bbox_inches='tight')
plt.show()